# DeepLab Archaeology Segmentation Runner

Рабочий Kaggle-runner для `03_multiclass_segmentation_deeplab`.

Notebook не содержит training logic: он только клонирует/обновляет репозиторий, ставит зависимости, проверяет окружение и запускает воспроизводимые scripts.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys


def run(cmd, cwd=None, env=None, check=True):
    cmd = [str(item) for item in cmd]
    print("\n$", " ".join(cmd))
    return subprocess.run(cmd, cwd=cwd, env=env, check=check)

print("python:", sys.version)
print("executable:", sys.executable)

## 1. GPU Check

DeepLab лучше запускать с GPU. Если здесь `cuda available: False`, включи Kaggle GPU: `Settings -> Accelerator -> GPU`.

In [ ]:
import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU is not enabled. Training will be very slow on CPU.")

## 2. Clone Or Update Repository

По умолчанию используется GitHub repo. Можно переопределить `REPO_URL` и `BRANCH` через Kaggle environment variables.

In [ ]:
REPO_URL = os.environ.get("REPO_URL", "https://github.com/MataNerdy/Geodata_Archaeology_CV.git")
BRANCH = os.environ.get("BRANCH", "main")
REPO_DIR = Path("/kaggle/working/Geodata_Archaeology_CV")
FORCE_RECLONE = os.environ.get("FORCE_RECLONE", "0") == "1"

if FORCE_RECLONE and REPO_DIR.exists():
    print("FORCE_RECLONE=1, removing old repo...")
    shutil.rmtree(REPO_DIR)

if REPO_DIR.exists():
    print("Repo already exists. Pulling latest changes...")
    run(["git", "fetch", "origin"], cwd=REPO_DIR)
    run(["git", "checkout", BRANCH], cwd=REPO_DIR)
    run(["git", "pull", "origin", BRANCH], cwd=REPO_DIR)
else:
    print("Cloning repo...")
    run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)])

SEG_DIR = REPO_DIR / "03_multiclass_segmentation_deeplab"
assert SEG_DIR.exists(), f"Segmentation module not found: {SEG_DIR}"

os.environ["PYTHONPATH"] = str(SEG_DIR) + os.pathsep + os.environ.get("PYTHONPATH", "")
print("SEG_DIR:", SEG_DIR)
print("PYTHONPATH prefix:", SEG_DIR)
run(["git", "log", "--oneline", "-3"], cwd=REPO_DIR)

## 3. Kaggle Hotfix Guard

Эта ячейка делает notebook устойчивым к старым копиям репозитория, где пакет назывался `datasets`. В актуальной версии он называется `arch_datasets`, чтобы не конфликтовать с HuggingFace/Kaggle package `datasets`.

In [ ]:
old_dir = SEG_DIR / "datasets"
new_dir = SEG_DIR / "arch_datasets"

if old_dir.exists() and not new_dir.exists():
    old_dir.rename(new_dir)
    print("Renamed datasets -> arch_datasets")

for py_file in SEG_DIR.rglob("*.py"):
    text = py_file.read_text()
    updated = text
    updated = updated.replace("from datasets.archaeology_dataset", "from arch_datasets.archaeology_dataset")
    updated = updated.replace('"datasets", "losses", "models", "utils"', '"arch_datasets", "losses", "models", "utils"')
    if updated != text:
        py_file.write_text(updated)
        print("Patched imports:", py_file.relative_to(SEG_DIR))

print("arch_datasets exists:", new_dir.exists())
print("archaeology_dataset.py exists:", (new_dir / "archaeology_dataset.py").exists())
assert new_dir.exists(), "arch_datasets directory is missing"
assert (new_dir / "archaeology_dataset.py").exists(), "archaeology_dataset.py is missing"

## 4. Install Dependencies

`segmentation_models_pytorch` обычно не установлен в Kaggle base image, поэтому ставим его явно.

In [ ]:
run([
    sys.executable, "-m", "pip", "install", "-q",
    "segmentation-models-pytorch",
    "timm",
    "pretrainedmodels",
    "efficientnet-pytorch",
    "opencv-python-headless",
    "shapely",
    "PyYAML",
])

import segmentation_models_pytorch as smp
print("segmentation_models_pytorch:", smp.__version__)

## 5. Dataset Paths

Для текущего Kaggle Dataset путь такой:

`/kaggle/input/datasets/matanerdy/kurgans-dataset/segmentation_dataset/segmentation_dataset`

In [ ]:
DATA_ROOT = Path(os.environ.get(
    "DATA_ROOT",
    "/kaggle/input/datasets/matanerdy/kurgans-dataset/segmentation_dataset/segmentation_dataset",
))
RUN_ROOT = Path(os.environ.get(
    "RUN_ROOT",
    str(SEG_DIR / "runs"),
))
RUN_ROOT.mkdir(parents=True, exist_ok=True)

print("DATA_ROOT:", DATA_ROOT)
print("RUN_ROOT:", RUN_ROOT)
assert (DATA_ROOT / "metadata.csv").exists(), f"metadata.csv not found: {DATA_ROOT}"
assert (DATA_ROOT / "images").is_dir(), f"images dir not found: {DATA_ROOT / 'images'}"
assert (DATA_ROOT / "masks").is_dir(), f"masks dir not found: {DATA_ROOT / 'masks'}"

import pandas as pd
meta = pd.read_csv(DATA_ROOT / "metadata.csv")
print("samples:", len(meta))
display(meta.head())
display(meta.groupby(["region", "modality"]).size().reset_index(name="samples").head(20))

## 6. Import And Compile Checks

Проверяем, что scripts видят локальные packages и не конфликтуют с внешним `datasets`.

In [ ]:
run([
    sys.executable, "-m", "py_compile",
    "scripts/train.py",
    "scripts/evaluate.py",
    "scripts/visualize_predictions.py",
    "scripts/threshold_sweep.py",
], cwd=SEG_DIR)

run([
    sys.executable, "-c",
    "import runpy; runpy.run_path('scripts/train.py', run_name='__not_main__'); print('train imports ok')",
], cwd=SEG_DIR)

## 7. Smoke Test

Минимальный запуск на 2 эпохи. Его стоит выполнить перед полной серией.

In [ ]:
RUN_SMOKE = os.environ.get("RUN_SMOKE", "1") == "1"

if RUN_SMOKE:
    run([
        sys.executable, "scripts/train.py",
        "--config", "configs/binary_kurgan.yaml",
        "--data-root", str(DATA_ROOT),
        "--out-dir", str(RUN_ROOT / "smoke_test"),
        "--epochs", "2",
        "--batch-size", "2",
        "--save-samples", "2",
    ], cwd=SEG_DIR)
else:
    print("RUN_SMOKE=0, skipping smoke test")

## 8. Minimal Experiment Series

Серия:

1. `binary_kurgan_li_resnet34`
2. `binary_kurgan_li_resnet50`
3. `binary_kurgan_li_threshold_sweep`
4. `kurgan_multiclass_li_resnet34`
5. `kurgan_multiclass_li_resnet50`

По умолчанию эта ячейка выключена. Поставь `RUN_FULL_SERIES=1`, когда smoke test прошел. Для class-weight sweep используй `RUN_MODE=multiclass_weight_sweep`; для 5-class серии используй `RUN_MODE=archaeology_5class`; для archaeology-aware object pipeline используй `RUN_MODE=archaeology_5class_competition`.

In [ ]:
RUN_FULL_SERIES = os.environ.get("RUN_FULL_SERIES", "0") == "1"
RUN_MODE = os.environ.get("RUN_MODE", "full")

if RUN_FULL_SERIES:
    run([
        "bash", "run_kaggle_experiments.sh",
    ], cwd=SEG_DIR, env={
        **os.environ,
        "DATA_ROOT": str(DATA_ROOT),
        "RUN_ROOT": str(RUN_ROOT),
        "PYTHON_BIN": sys.executable,
        "RUN_MODE": RUN_MODE,
    })
else:
    print("RUN_FULL_SERIES=0, skipping experiment series")
    print("Set RUN_FULL_SERIES=1 and RUN_MODE=full, RUN_MODE=multiclass_weight_sweep, RUN_MODE=archaeology_5class, or RUN_MODE=archaeology_5class_competition.")


## 9. Show Results

Показывает найденные `evaluation.csv`, `threshold_sweep.json` и prediction examples, если они уже созданы.

In [ ]:
from IPython.display import Image, display

for eval_path in sorted(RUN_ROOT.rglob("evaluation.csv")):
    print("\n", eval_path.relative_to(RUN_ROOT))
    display(pd.read_csv(eval_path))

for sweep_path in sorted(RUN_ROOT.rglob("threshold_sweep.json")):
    print("\n", sweep_path.relative_to(RUN_ROOT))
    payload = json.loads(sweep_path.read_text())
    print(json.dumps(payload.get("best", payload), indent=2, ensure_ascii=False))

for image_path in sorted(RUN_ROOT.rglob("prediction_examples.png"))[:5]:
    print("\n", image_path.relative_to(RUN_ROOT))
    display(Image(filename=str(image_path)))

## 10. Zip Runs

Архив для скачивания из Kaggle output.

In [ ]:
zip_path = Path("/kaggle/working/deeplab_runs.zip")
if zip_path.exists():
    zip_path.unlink()
run(["zip", "-r", str(zip_path), str(RUN_ROOT)])
print("Saved:", zip_path)